pip install 하려면 <br>
현재 분석 환경 경로 -> 터미널 열기 -> 가상환경 실행 -> uv pip install ~~

In [20]:
# import pandas as pd
import polars as pl

# ===== 경로 설정 =====
DATA_DIR = "../data/raw"

# ① 데이터 정제 + 평가셋 라벨링

**목표:** 1~2점 + 코멘트 있는 리뷰를 정제하고, 사람이 검수한 정답셋(`eval.csv`)과 예시 풀(`pool.csv`)을 만든다.

---

## 1. 정제
- [ ] 6개 테이블 조인 (reviews, orders, order_items, products + translation, sellers, customers)
- [ ] 대상 필터링: 1~2점 & 코멘트 있음
- [ ] 중복 제거: `(review_id, order_id)` 기준
- [ ] 판매자 귀속 규칙 결정 (단일 판매자 주문만 / 관련 판매자 전원 귀속)
- [ ] 텍스트 정리: 제목 + 본문 합치기, 공백 정리 (포르투갈어 원문 유지)

## 2. 라벨 체계 정의 ⭐ 가장 중요
- [ ] 카테고리 확정 (예: 배송지연 / 미수령 / 오배송·누락 / 품질·파손 / 설명과 다름 / 응대·환불 / 기타)
- [ ] 카테고리별 정의, 경계 규칙, 애매한 사례 문서화
- [ ] 단일 라벨(주 원인 하나)로 시작

## 3. 라벨링
- [ ] 파일럿 50건 → 가이드라인 수정
- [ ] 평가셋 300~500건 (층화 샘플, 성능 측정 전용)
- [ ] 예시 풀 200~500건 (few-shot / RAG용) — **평가셋과 절대 겹치지 않게**
- [ ] 일관성 점검: 일부를 며칠 뒤 재라벨링 → kappa 확인
- [ ] 배송지연 라벨을 실제 배송일과 대조
      (`order_delivered_customer_date > order_estimated_delivery_date`)

---

## 주의사항
- 코멘트 없는 리뷰가 절반 이상
- `review_id`는 유일하지 않음
- 한 주문에 판매자가 여러 명일 수 있음
- 평가셋은 LLM 선라벨링 금지 → 사람이 검수한 라벨 기준

## 산출물
- [ ] 정제 parquet
- [ ] 라벨 가이드 문서
- [ ] `eval.csv`
- [ ] `pool.csv`

# EDA

- 데이터 구조

<!-- ![데이터 구조](data/raw/data_structure.png) -->
<img src="../data/data_structure.png" width="800">

- olist_order_reviews_dataset
  - review_id, order_id
  - review_score, review_comment_title, review_comment_message, review_creation_date, review_answer_timestamp
- olist_orders_dataset
  - order_id, customer_id
  - order_status, order_purchase_timestamp, order_approved_at, order_delivered_carrier_date, order_delivered_customer_date,order_estimated_delivery_date
- olist_order_payments_dataset
  - order_id
  - payment_sequential, payment_type, payment_installments, payment_value
- olist_customers_dataset
  - customer_id, customer_unique_id, customer_zip_code_prefix
  - customer_city, customer_state
- olist_geolocation_dataset
  - geolocation_zip_code_prefix
  - geolocation_lat, geolocation_lng, geolocation_city, geolocation_state
- olist_order_items_dataset
  - order_id, order_item_id, product_id, seller_id
  - shipping_limit_date, price, freight_value
- olist_products_dataset
  - product_id
  - product_category_name, product_name_lenght, product_description_lenght, product_photos_qty, product_weight_g, product_length_cm, product_height_cm, product_width_cm
- olist_sellers_dataset
  - seller_id, seller_zip_code_prefix
  - seller_city, seller_state

In [18]:
df = pl.read_csv("../data/raw/olist_sellers_dataset.csv")
# df.head()
df.columns

['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']

In [ ]:
# 활용법
df = pl.read_csv(
    "data.csv",
    separator=",",           # 구분자 (pandas의 sep)
    has_header=True,         # 헤더 여부
    columns=["a", "b"],      # 필요한 컬럼만
    schema_overrides={"id": pl.Utf8},  # 특정 컬럼 타입 지정
    encoding="utf8",         # 한글 CP949 파일이면 "cp949"는 미지원 → 아래 참고
)

## lazy로 불러오기
lf = pl.scan_csv("data.csv")   # LazyFrame — 아직 안 읽음
df = lf.filter(pl.col("a") > 10).select("a", "b").collect()  # 이때 실제로 읽음

# 정제

- [ ] 대상 필터링: 1~2점 & 코멘트 있음
- [ ] 중복 제거: `(review_id, order_id)` 기준
- [ ] 판매자 귀속 규칙 결정 (단일 판매자 주문만 / 관련 판매자 전원 귀속)
- [ ] 텍스트 정리: 제목 + 본문 합치기, 공백 정리 (포르투갈어 원문 유지)

In [26]:
reviews_raw = pl.read_csv(f"{DATA_DIR}/olist_order_reviews_dataset.csv", try_parse_dates=True)
reviews_raw.head()

review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
str,str,i64,str,str,datetime[μs],datetime[μs]
"""7bc2406110b926393aa56f80a40eba…","""73fc7af87114b39712e6da79b0a377…",4,null,null,2018-01-18 00:00:00,2018-01-18 21:46:59
"""80e641a11e56f04c1ad469d5645fdf…","""a548910a1c6147796b98fdf73dbeba…",5,null,null,2018-03-10 00:00:00,2018-03-11 03:05:13
"""228ce5500dc1d8e020d8d1322874b6…","""f9e4b658b201a9f2ecdecbb34bed03…",5,null,null,2018-02-17 00:00:00,2018-02-18 14:36:24
"""e64fb393e7b32834bb789ff8bb3075…","""658677c97b385a9be170737859d351…",5,null,"""Recebi bem antes do prazo esti…",2017-04-21 00:00:00,2017-04-21 22:02:06
"""f7c4243c7fe1938f181bec41a392bd…","""8e6bfb81e283fa7e4f11123a3fb894…",5,null,"""Parabéns lojas lannister adore…",2018-03-01 00:00:00,2018-03-02 10:26:53


### 대상 필터링
- 점수 1~2점, 코멘트 있는 리뷰

In [27]:
# ===== 필터링: 1~2점 & 코멘트 본문 있음 =====
# 공백만 있는 코멘트도 빈 것으로 간주
reviews_neg = reviews_raw.filter(
    pl.col("review_score").is_in([1, 2])
    & pl.col("review_comment_message").is_not_null()
    & (pl.col("review_comment_message").str.strip_chars() != "")
)

# ===== 결과 확인 =====
print(f"전체 리뷰: {reviews_raw.height:,}건")
print(f"필터링 후: {reviews_neg.height:,}건")
reviews_neg.head()

전체 리뷰: 99,224건
필터링 후: 10,889건


review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
str,str,i64,str,str,datetime[μs],datetime[μs]
"""9314d6f9799f5bfba510cc7bcd468c…","""0dacf04c5ad59fd5a0cc1faa07c34e…",2,null,"""GOSTARIA DE SABER O QUE HOUVE,…",2018-01-18 00:00:00,2018-01-20 21:25:45
"""373cbeecea8286a2b66c97b1b157ec…","""583174fbe37d3d5f0d6661be3aad17…",1,"""Não chegou meu produto ""","""Péssimo""",2018-08-15 00:00:00,2018-08-15 04:10:37
"""2c5e27fc178bde7ac173c9c62c31b0…","""0ce9a24111d850192a933fcaab6fba…",1,null,"""Não gostei ! Comprei gato por …",2017-12-13 00:00:00,2017-12-16 07:14:07
"""58044bca115705a48fe0e00a21390c…","""68e55ca79d04a79f20d4bfc0146f4b…",1,null,"""Sempre compro pela Internet e …",2018-04-08 00:00:00,2018-04-09 12:22:39
"""9fd59cd04b42f600df9f25e54082a8…","""3c314f50bc654f3c4e317b055681df…",1,null,"""Nada de chegar o meu pedido.""",2017-04-21 00:00:00,2017-04-23 05:37:03


### 중복 제거
- review_id, order_id가 같으면 최신 리뷰만 남김

In [28]:
# ===== 같은 (review_id, order_id) 쌍이 2번 이상인 행 추출 =====
# 쌍별 등장 횟수를 컬럼으로 붙인 뒤 2 이상만 남김
pair_dup_rows = (
    reviews_neg
    .with_columns(pl.len().over(["review_id", "order_id"]).alias("pair_count"))
    .filter(pl.col("pair_count") > 1)
    .sort(["review_id", "order_id", "review_creation_date"])
    .select([
        "review_id",
        "order_id",
        "pair_count",
        "review_score",
        "review_creation_date",
        "review_answer_timestamp",
        "review_comment_title",
        "review_comment_message",
    ])
)

# ===== 결과 확인 =====
# 긴 코멘트가 잘리지 않도록 출력 설정을 이 블록에서만 변경
print(f"중복 쌍 행 수: {pair_dup_rows.height:,}건")
with pl.Config(fmt_str_lengths=200, tbl_rows=50):
    display(pair_dup_rows)

중복 쌍 행 수: 0건


review_id,order_id,pair_count,review_score,review_creation_date,review_answer_timestamp,review_comment_title,review_comment_message
str,str,u32,i64,datetime[μs],datetime[μs],str,str


In [29]:
# ===== 중복 현황 확인 =====
# 쌍 기준 중복: 완전히 같은 (review_id, order_id)가 여러 행
pair_dup_count = reviews_neg.height - reviews_neg.n_unique(subset=["review_id", "order_id"])
# review_id 기준 중복: 같은 리뷰가 여러 주문에 연결된 경우
review_id_dup_count = reviews_neg.height - reviews_neg["review_id"].n_unique()

print(f"(review_id, order_id) 중복: {pair_dup_count:,}건")
print(f"review_id만 중복: {review_id_dup_count:,}건")

# ===== (review_id, order_id) 기준 중복 제거 =====
# 최신 작성일이 위로 오도록 정렬 후 첫 행만 유지
reviews_dedup = (
    reviews_neg
    .sort("review_creation_date", descending=True)
    .unique(subset=["review_id", "order_id"], keep="first", maintain_order=True)
)

# ===== 결과 확인 =====
print(f"중복 제거 전: {reviews_neg.height:,}건")
print(f"중복 제거 후: {reviews_dedup.height:,}건")

(review_id, order_id) 중복: 0건
review_id만 중복: 131건
중복 제거 전: 10,889건
중복 제거 후: 10,889건


### 판매자 귀속 규칙 및 라벨링 참고 정보 생성
- 주문 하나 당 리뷰 하나가 있는데, 한 주문에 제품 여럿 판매자 여럿이면 어떤 판매자에게 불만을 보내야하는지 애매
- 판매자 귀속 규칙 결정 → 분류는 전체, 판매자 집계는 단일 판매자 주문만 (`seller_count == 1`, 91.11%)

- 주문별 판매자 수 확인

In [30]:
# ===== 주문 상품 데이터 불러오기 =====
order_items_path = f"{DATA_DIR}/olist_order_items_dataset.csv"
order_items = pl.read_csv(order_items_path, try_parse_dates=True)

# ===== 주문별 판매자 수 계산 =====
# 같은 판매자의 상품이 여러 개여도 1명으로 셈
sellers_per_order = (
    order_items
    .group_by("order_id")
    .agg(pl.col("seller_id").n_unique().alias("seller_count"))
)

# ===== 필터링된 리뷰에 판매자 수 붙이기 =====
# left join: order_items에 없는 주문(취소 등)은 seller_count가 null
reviews_seller = reviews_dedup.join(sellers_per_order, on="order_id", how="left")

# ===== 판매자 수 분포 확인 =====
seller_count_dist = (
    reviews_seller
    .group_by("seller_count")
    .agg(pl.len().alias("review_count"))
    .with_columns(
        (pl.col("review_count") / pl.col("review_count").sum() * 100)
        .round(2)
        .alias("ratio_pct")
    )
    .sort("seller_count", nulls_last=True)
)
seller_count_dist

seller_count,review_count,ratio_pct
u32,u32,f64
1,9921,91.11
2,483,4.44
3,27,0.25
4,3,0.03
5,2,0.02
null,453,4.16


- 라벨링 참고 정보
  - 배송 지연
  - 주문 상태

### 텍스트 정리
- 제목, 본문 합치기
- 

# 라벨 체계 정의

- [ ] 카테고리 확정 (예: 배송지연 / 미수령 / 오배송·누락 / 품질·파손 / 설명과 다름 / 응대·환불 / 기타)
- [ ] 카테고리별 정의, 경계 규칙, 애매한 사례 문서화
- [ ] 단일 라벨(주 원인 하나)로 시작

# 라벨링

- [ ] 파일럿 50건 → 가이드라인 수정
- [ ] 평가셋 300~500건 (층화 샘플, 성능 측정 전용)
- [ ] 예시 풀 200~500건 (few-shot / RAG용) — **평가셋과 절대 겹치지 않게**
- [ ] 일관성 점검: 일부를 며칠 뒤 재라벨링 → kappa 확인
- [ ] 배송지연 라벨을 실제 배송일과 대조
      (`order_delivered_customer_date > order_estimated_delivery_date`)